In [29]:
from clearml import Task, Dataset
import pandas as pd
import joblib

from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score

In [30]:
task = Task.init(
    project_name="Course Work",
    task_name="Model CatBoost"
)

ClearML Task: created new task id=1560cfab46564f5e9287dc2d1c8174bb


Could not fetch GPU stats: NVML Shared Library Not Found


ClearML results page: https://app.clear.ml/projects/61047c7a66aa40eab8cd971dc2d8b828/experiments/1560cfab46564f5e9287dc2d1c8174bb/output/log


ClearML Monitor: GPU monitoring failed getting GPU reading, switching off GPU monitoring


In [31]:
dataset = Dataset.get(
    dataset_name="Transactions Features",
    dataset_project="Course Work",
    only_completed=True
)

data_path = dataset.get_local_copy()

train = pd.read_csv(f"{data_path}/train.csv")
test = pd.read_csv(f"{data_path}/test.csv")

In [32]:
# Подготовка признаков
feature_cols = [col for col in train.columns if col not in ["clientID", "target"]]
X_train = train[feature_cols]
y_train = train["target"]
X_test = test[feature_cols]
y_test = test["target"]

In [33]:
# Гиперпараметры
params = {
    'iterations': 100,
    'depth': 4,
    'learning_rate': 0.1,
    'random_seed': 42,
    'verbose': 0
}
task.connect(params)

{'iterations': 100,
 'depth': 4,
 'learning_rate': 0.1,
 'random_seed': 42,
 'verbose': 0}

In [34]:
vmodel = CatBoostClassifier(**params)
vmodel.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
    verbose=False,
    use_best_model=True
)

CatBoostClassifier(depth=4, iterations=100, learning_rate=0.1, random_seed=42, verbose=0)

In [35]:
evals_result = vmodel.get_evals_result()

In [36]:
if 'learn' in evals_result and 'Logloss' in evals_result['learn']:
    for i, loss in enumerate(evals_result['learn']['Logloss']):
        task.get_logger().report_scalar(
            title="Logloss", 
            series="train", 
            value=loss, 
            iteration=i
        )

In [38]:
y_train_proba = vmodel.predict_proba(X_train)[:, 1]
y_test_proba  = vmodel.predict_proba(X_test)[:, 1]
auc_train = roc_auc_score(y_train, y_train_proba)
auc_test  = roc_auc_score(y_test, y_test_proba)

print(f"ROC-AUC (train): {auc_train:.4f}")
print(f"ROC-AUC (test) : {auc_test:.4f}")

# Логирование в ClearML
task.get_logger().report_scalar("ROC-AUC", "train", auc_train, 0)
task.get_logger().report_scalar("ROC-AUC", "test",  auc_test, 0)

ROC-AUC (train): 0.8412
ROC-AUC (test) : 0.8323


In [39]:
task.get_logger().report_scalar(
    title="ROC-AUC", 
    series="test", 
    value=auc_roc, 
    iteration=params['iterations']
)

In [40]:
joblib.dump(vmodel, "model_catboost.pkl")
task.upload_artifact("model_catboost", "model_catboost.pkl")

True

In [41]:
print("Эксперимент завершен")
task.close()

Эксперимент завершен
